In [4]:
import time
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from uncertainties import ufloat
from ctypes import c_double
from pathlib import Path
import pickle
import os

import ROOT
ROOT.EnableImplicitMT()    # Tells ROOT to go parallel

In [ ]:
def makeTables(
        xedges: np.ndarray,
        yedges: np.ndarray,
        bkg_hist: ROOT.TH2,
        sig_hist: ROOT.TH2,
        x_loCut: float = 0.0,
        y_loCut: float = 0.0,
        sigScale: float = 1.0,
        bkgScale: float = 1.0,
):
    xedges = np.round(np.asarray(xedges, dtype=np.float64), 8)
    yedges = np.round(np.asarray(yedges, dtype=np.float64), 8)

    xidx = pd.Index(xedges)# , name="x_cut")
    yidx = pd.Index(yedges)# , name="y_cut")

    empty_df = pd.DataFrame(index=xidx, columns=yidx, dtype=float)

    # Create empty tables
    tableNames = ['sig_NA', 'sig_NB', 'sig_NC', 'sig_ND',
                  'bkg_NA', 'bkg_NB', 'bkg_NC', 'bkg_ND',
                  'sig_NA_unc', 'sig_NB_unc', 'sig_NC_unc', 'sig_ND_unc',
                  'bkg_NA_unc', 'bkg_NB_unc', 'bkg_NC_unc', 'bkg_ND_unc',
                  'Z_A', 'Z_B', 'Z_C', 'Z_D',
                  'Z_noncl_A', 'Z_noncl_B', 'Z_noncl_C', 'Z_noncl_D',
                  # 'Z_noncl_plus1s_A', 'Z_noncl_plus1s_B', 'Z_noncl_plus1s_C', 'Z_noncl_plus1s_D',
                  'noncl', 'noncl_unc',
                  'delta_A', 'delta_A_unc',
                ]
    tables = dict()
    for name in tableNames:
        tables[name] = empty_df.copy()
     
    for x_boundary in xidx.to_numpy():
        for y_boundary in yidx.to_numpy():
            x_lo    = bkg_hist.GetXaxis().FindBin(x_loCut)
            x_up    = bkg_hist.GetNbinsX()+1
            x_bound = bkg_hist.GetXaxis().FindBin(x_boundary)

            y_lo    = bkg_hist.GetYaxis().FindBin(y_loCut)
            y_up    = bkg_hist.GetNbinsY()+1
            y_bound = bkg_hist.GetYaxis().FindBin(y_boundary)

            if x_boundary <= x_loCut: continue
            if y_boundary <= y_loCut: continue

            # DEBUG
            # -----------------------------------
            # print('x_loCut', x_loCut)
            # print('y_loCut', y_loCut)
            # print('x_boundary:', x_boundary)
            # print('y_boundary:', y_boundary)
            # print('x_lo:', x_lo)
            # print('x_bound:', x_bound)
            # print('x_up:', x_up)
            # print('y_lo:', y_lo)
            # print('y_bound:', y_bound)
            # print('y_up:', y_up)
            # exit()


            # ------------ Backgrounds --------------
            c_err = c_double(0.0)
            bkg_NA = ufloat(bkg_hist.IntegralAndError(x_bound,    x_up,       y_bound,    y_up,          c_err), c_err.value) * bkgScale
            bkg_NB = ufloat(bkg_hist.IntegralAndError(x_lo,       x_bound-1,  y_bound,    y_up,          c_err), c_err.value) * bkgScale
            bkg_NC = ufloat(bkg_hist.IntegralAndError(x_bound,    x_up,       y_lo,      y_bound-1,      c_err), c_err.value) * bkgScale
            bkg_ND = ufloat(bkg_hist.IntegralAndError(x_lo,       x_bound-1,  y_lo,      y_bound-1,      c_err), c_err.value) * bkgScale

            num   = bkg_NB * bkg_NC
            denom = bkg_NA * bkg_ND
            noncl = np.abs(1- num/denom) if denom.n > 0 else ufloat(1., 1.) # Check this line!

            delta_A = bkg_NA - (bkg_NB * bkg_NC) / bkg_ND if bkg_ND > 0 else ufloat(float('nan'),float('nan'))

            tables['bkg_NA'].loc[x_boundary, y_boundary] = bkg_NA.n
            tables['bkg_NB'].loc[x_boundary, y_boundary] = bkg_NB.n
            tables['bkg_NC'].loc[x_boundary, y_boundary] = bkg_NC.n
            tables['bkg_ND'].loc[x_boundary, y_boundary] = bkg_ND.n

            tables['bkg_NA_unc'].loc[x_boundary, y_boundary] = bkg_NA.s
            tables['bkg_NB_unc'].loc[x_boundary, y_boundary] = bkg_NB.s
            tables['bkg_NC_unc'].loc[x_boundary, y_boundary] = bkg_NC.s
            tables['bkg_ND_unc'].loc[x_boundary, y_boundary] = bkg_ND.s

            tables['noncl'].loc[x_boundary, y_boundary]     = noncl.n
            tables['noncl_unc'].loc[x_boundary, y_boundary] = noncl.s

            tables['delta_A'].loc[x_boundary, y_boundary]     = delta_A.n
            tables['delta_A_unc'].loc[x_boundary, y_boundary] = delta_A.s
            

            # ------------ Signals --------------
            c_err = c_double(0.0)
            sig_NA = ufloat(sig_hist.IntegralAndError(x_bound,    x_up,       y_bound,    y_up,          c_err), c_err.value) * sigScale
            sig_NB = ufloat(sig_hist.IntegralAndError(x_lo,       x_bound-1,  y_bound,    y_up,          c_err), c_err.value) * sigScale
            sig_NC = ufloat(sig_hist.IntegralAndError(x_bound,    x_up,       y_lo,      y_bound-1,      c_err), c_err.value) * sigScale
            sig_ND = ufloat(sig_hist.IntegralAndError(x_lo,       x_bound-1,  y_lo,      y_bound-1,      c_err), c_err.value) * sigScale

            tables['sig_NA'].loc[x_boundary, y_boundary] = sig_NA.n
            tables['sig_NB'].loc[x_boundary, y_boundary] = sig_NB.n
            tables['sig_NC'].loc[x_boundary, y_boundary] = sig_NC.n
            tables['sig_ND'].loc[x_boundary, y_boundary] = sig_ND.n

            tables['sig_NA_unc'].loc[x_boundary, y_boundary] = sig_NA.s
            tables['sig_NB_unc'].loc[x_boundary, y_boundary] = sig_NB.s
            tables['sig_NC_unc'].loc[x_boundary, y_boundary] = sig_NC.s
            tables['sig_ND_unc'].loc[x_boundary, y_boundary] = sig_ND.s

            # ------------ Significance --------------
            eps = 5e-1

            Z_A = ROOT.RooStats.AsimovSignificance(sig_NA.n, max(eps, bkg_NA.n), calc_unc(max(eps, bkg_NA.n), max(eps, bkg_NA.s), 0.))
            Z_B = ROOT.RooStats.AsimovSignificance(sig_NB.n, max(eps, bkg_NB.n), calc_unc(max(eps, bkg_NB.n), max(eps, bkg_NB.s), 0.))
            Z_C = ROOT.RooStats.AsimovSignificance(sig_NC.n, max(eps, bkg_NC.n), calc_unc(max(eps, bkg_NC.n), max(eps, bkg_NC.s), 0.))
            Z_D = ROOT.RooStats.AsimovSignificance(sig_ND.n, max(eps, bkg_ND.n), calc_unc(max(eps, bkg_ND.n), max(eps, bkg_ND.s), 0.))

            
            tables['Z_A'].loc[x_boundary, y_boundary] = Z_A
            tables['Z_B'].loc[x_boundary, y_boundary] = Z_B
            tables['Z_C'].loc[x_boundary, y_boundary] = Z_C
            tables['Z_D'].loc[x_boundary, y_boundary] = Z_D

            # ------------ Significance with non-clsoure uncertainty --------------

            Z_noncl_A = ROOT.RooStats.AsimovSignificance(sig_NA.n, max(eps, bkg_NA.n), calc_unc(max(eps, bkg_NA.n), max(eps, bkg_NA.s), abs(noncl.n)))
            Z_noncl_B = ROOT.RooStats.AsimovSignificance(sig_NB.n, max(eps, bkg_NB.n), calc_unc(max(eps, bkg_NB.n), max(eps, bkg_NB.s), abs(noncl.n)))
            Z_noncl_C = ROOT.RooStats.AsimovSignificance(sig_NC.n, max(eps, bkg_NC.n), calc_unc(max(eps, bkg_NC.n), max(eps, bkg_NC.s), abs(noncl.n)))
            Z_noncl_D = ROOT.RooStats.AsimovSignificance(sig_ND.n, max(eps, bkg_ND.n), calc_unc(max(eps, bkg_ND.n), max(eps, bkg_ND.s), abs(noncl.n)))

            
            tables['Z_noncl_A'].loc[x_boundary, y_boundary] = Z_noncl_A
            tables['Z_noncl_B'].loc[x_boundary, y_boundary] = Z_noncl_B
            tables['Z_noncl_C'].loc[x_boundary, y_boundary] = Z_noncl_C
            tables['Z_noncl_D'].loc[x_boundary, y_boundary] = Z_noncl_D

            # ------------ Significance with non-clsoure uncertainty plus one sigma unc. -------

            # Z_noncl_plus1s_A = ROOT.RooStats.AsimovSignificance(max(eps, sig_NA), max(eps, bkg_NA.n), calc_unc(max(eps, bkg_NA.n), max(eps, bkg_NA.s), noncl.n))
            # Z_noncl_plus1s_B = ROOT.RooStats.AsimovSignificance(max(eps, sig_NB), max(eps, bkg_NB.n), calc_unc(max(eps, bkg_NB.n), max(eps, bkg_NB.s), noncl.n))
            # Z_noncl_plus1s_C = ROOT.RooStats.AsimovSignificance(max(eps, sig_NC), max(eps, bkg_NC.n), calc_unc(max(eps, bkg_NC.n), max(eps, bkg_NC.s), noncl.n))
            # Z_noncl_plus1s_D = ROOT.RooStats.AsimovSignificance(max(eps, sig_ND), max(eps, bkg_ND.n), calc_unc(max(eps, bkg_ND.n), max(eps, bkg_ND.s), noncl.n))

            
            # tables['Z_noncl_plus1s_A'].loc[x_boundary, y_boundary] = Z_A
            # tables['Z_noncl_plus1s_B'].loc[x_boundary, y_boundary] = Z_B
            # tables['Z_noncl_plus1s_C'].loc[x_boundary, y_boundary] = Z_C
            # tables['Z_noncl_plus1s_D'].loc[x_boundary, y_boundary] = Z_D
            
    return tables

In [5]:
HISTDIR = Path('/scratch-cbe/users/alikaan.gueven/AN_plots/ParT_hists/AN-25-092_ML_plots_limitcalc_merge_w_Ang_v3/run2plus3')
PLANEs = ['GT0', 'GT1', 'GT2', 'GT3']

sig_file = ROOT.TFile(os.path.join(HISTDIR, "stop_M1000_980_ct2_run2plus3_hist.root"))
sig_hist = sig_file.all_evt.MET_pt_corr_vs_Sleadingvtx_MLscore.Clone()


bkg_file = ROOT.TFile(os.path.join(HISTDIR, f"data_run2plus3_hist.root"))
bkg_hist = bkg_file.all_evt.MET_pt_corr_vs_leadingvtx_MLscore.Clone()

AttributeError: <class cppyy.gbl.TFile at 0x55d37b9b53f0> object has no attribute 'all_evt'

Error in <TList::Clear>: A list is accessing an object (0x55d37bbc8bf0) already deleted (list name = TList)
Error in <TList::Clear>: A list is accessing an object (0x55d37bc1c380) already deleted (list name = TList)
Error in <TList::Clear>: A list is accessing an object (0x55d37bc23fb0) already deleted (list name = TList)
Error in <TList::Clear>: A list is accessing an object (0x55d37bc24320) already deleted (list name = TList)
Error in <TList::Clear>: A list is accessing an object (0x55d37bc24690) already deleted (list name = TList)
Error in <TList::Clear>: A list is accessing an object (0x55d37bc24a00) already deleted (list name = TList)
Error in <TList::Clear>: A list is accessing an object (0x55d37bc24f30) already deleted (list name = TList)
Error in <TList::Clear>: A list is accessing an object (0x55d37bc253d0) already deleted (list name = TList)
Error in <TList::Clear>: A list is accessing an object (0x55d37bc257a0) already deleted (list name = TList)
Error in <TList::Clear>: A l